In [4]:
import pandas as pd
import lxml
from pathlib import Path
from glob import glob

In [9]:
bases = ["CAPP", "CASS", "CONSTIT", "INCA", "JADE"]
root = "/mnt/data/Murphy/src"

paths = {
    base: glob(f"{root}/{base}/**/*.*", recursive=True)
    for base in bases
}

for base, files in paths.items():
    print(f"{base}: {len(files)} fichiers")

CAPP: 1 fichiers
CASS: 92 fichiers
CONSTIT: 2 fichiers
INCA: 1 fichiers
JADE: 256 fichiers


In [28]:
import math
from glob import glob
from collections import defaultdict
import xml.etree.ElementTree as ET


def strip_ns(tag):
    return tag.rsplit('}', 1)[-1] if '}' in tag else tag


class Node:
    __slots__ = ("name", "occ", "sumsq", "min_k", "max_k", "n_present", "children")

    def __init__(self, name):
        self.name = name
        self.occ = 0          # total d'instances de l'élément (= somme des k)
        self.sumsq = 0        # somme des k^2  (pour std)
        self.min_k = None     # min de k sur les instances de parent où présent
        self.max_k = 0
        self.n_present = 0    # nb d'instances de parent contenant >=1 de cet enfant
        self.children = {}

    def child(self, name):
        c = self.children.get(name)
        if c is None:
            c = Node(name)
            self.children[name] = c
        return c

    def observe(self, k):
        # appelé une fois par instance de parent ; k = nb d'enfants de ce nom (k>=1)
        self.occ += k
        self.sumsq += k * k
        self.n_present += 1
        if k > self.max_k:
            self.max_k = k
        if self.min_k is None or k < self.min_k:
            self.min_k = k


def _merge(elem, node, strip):
    groups = defaultdict(list)
    for c in elem:
        t = strip_ns(c.tag) if strip else c.tag
        groups[t].append(c)
    for name, lst in groups.items():
        cn = node.child(name)
        cn.observe(len(lst))
        for c in lst:
            _merge(c, cn, strip)


def _doc_entries(root, split_level, strip):
    # -> dict: key_tuple -> liste des éléments d'entrée pour ce document
    if split_level == 0:
        return {(): [root]}
    res = defaultdict(list)

    def rec(elem, path):
        t = strip_ns(elem.tag) if strip else elem.tag
        newpath = path + (t,)
        if len(newpath) == split_level:
            res[newpath].append(elem)
        else:
            for c in elem:
                rec(c, newpath)

    rec(root, ())
    return res


def aggregate_base(paths, split_level=0, strip_namespaces=True):
    forest = {}          # key tuple -> Node racine
    ndocs = 0
    for p in paths:
        try:
            root = ET.parse(p).getroot()
        except ET.ParseError:
            continue
        ndocs += 1

        if split_level == 0:
            # la racine XML EST le nœud racine de l'arbre ; clé = son nom
            rtag = strip_ns(root.tag) if strip_namespaces else root.tag
            key = (rtag,)
            rnode = forest.get(key)
            if rnode is None:
                rnode = Node(rtag)
                forest[key] = rnode
            rnode.observe(1)               # une instance de racine par document
            _merge(root, rnode, strip_namespaces)
        else:
            for key, entries in _doc_entries(root, split_level, strip_namespaces).items():
                rnode = forest.get(key)
                if rnode is None:
                    rnode = Node(key[-1])
                    forest[key] = rnode
                rnode.observe(len(entries))
                for e in entries:
                    _merge(e, rnode, strip_namespaces)
    return forest, ndocs


def _serialize(node, n_parent):
    # n_parent = nb d'instances de parent (zéros inclus)
    avg = node.occ / n_parent if n_parent else 0.0
    var = max(node.sumsq / n_parent - avg * avg, 0.0) if n_parent else 0.0
    mn = node.min_k if node.n_present == n_parent else 0   # absent qq part -> min 0
    return {
        "name": node.name,
        "occ": node.occ,
        "min": mn if mn is not None else 0,
        "max": node.max_k,
        "avg": avg,
        "std": math.sqrt(var),
        "children": [
            _serialize(c, node.occ)
            for c in sorted(node.children.values(), key=lambda x: -x.occ)
        ],
    }


def analyze(root_dir, bases, split_level=0, strip_namespaces=True):
    out = {}
    for base in bases:
        paths = glob(f"{root_dir}/{base}/**/*.xml", recursive=True)
        forest, ndocs = aggregate_base(paths, split_level, strip_namespaces)
        out[base] = {
            "ndocs": ndocs,
            "trees": [
                {"key": list(k), **_serialize(n, ndocs)}
                for k, n in forest.items()
            ],
        }
    return out

In [37]:
def _cardinality(node, n_parent):
    """Renvoie un symbole de cardinalité type-schéma à partir des stats."""
    optional = node["min"] == 0           # absent dans au moins une instance de parent
    multiple = node["max"] > 1
    if optional and multiple:
        return "*"   # 0..n
    if optional:
        return "?"   # 0..1
    if multiple:
        return "+"   # 1..n
    return ""        # 1..1 (obligatoire, singleton) -> pas de marqueur


def _format_line(node, prefix, is_last):
    branch = "└─ " if is_last else "├─ "
    card = _cardinality(node, None)
    name = node["name"]
    occ = node["occ"]

    # affichage adaptatif : stats détaillées seulement si non-trivial
    nontrivial = node["max"] > 1 or node["min"] == 0
    if nontrivial:
        stats = f"  [{node['min']}–{node['max']}] ~{node['avg']:.2f} ±{node['std']:.2f}"
    else:
        stats = ""

    flag = f" {card}" if card else ""
    return f"{prefix}{branch}{name}{flag}  ×{occ}{stats}"


def print_tree(tree, _prefix="", _is_root=True):
    """Affiche un arbre (un dict racine renvoyé par _serialize/analyze)."""
    if _is_root:
        # racine : pas de branche, juste le nom
        card = _cardinality(tree, None)
        flag = f" {card}" if card else ""
        nontrivial = tree["max"] > 1 or tree["min"] == 0
        stats = (f"  [{tree['min']}–{tree['max']}] ~{tree['avg']:.2f} ±{tree['std']:.2f}"
                 if nontrivial else "")
        print(f"{tree['name']}{flag}  ×{tree['occ']}{stats}")
        children = tree["children"]
        for i, c in enumerate(children):
            print_tree(c, "", i == len(children) - 1)
        return

    print(_format_line(tree, _prefix, _is_root))  # placeholder, voir ci-dessous


def print_base(base_result, base_name=""):
    """Affiche tous les arbres d'une base (sortie de analyze()[base])."""
    print(f"### {base_name}  ({base_result['ndocs']} documents)\n")
    for tree in base_result["trees"]:
        key = "/".join(tree["key"]) or "<base>"
        if tree["key"]:
            print(f"— arbre: {key} —")
        print_tree(tree)
        print()

In [38]:
def _stats_suffix(node):
    if node["max"] > 1 or node["min"] == 0:
        return f"  [{node['min']}–{node['max']}] ~{node['avg']:.2f} ±{node['std']:.2f}"
    return ""

def _card_flag(node):
    optional, multiple = node["min"] == 0, node["max"] > 1
    sym = "*" if (optional and multiple) else "?" if optional else "+" if multiple else ""
    return f" {sym}" if sym else ""

def print_tree(node, prefix="", is_last=True, is_root=True):
    if is_root:
        print(f"{node['name']}{_card_flag(node)}  ×{node['occ']}{_stats_suffix(node)}")
    else:
        branch = "└─ " if is_last else "├─ "
        print(f"{prefix}{branch}{node['name']}{_card_flag(node)}  ×{node['occ']}{_stats_suffix(node)}")
        prefix += "   " if is_last else "│  "
    kids = node["children"]
    for i, c in enumerate(kids):
        print_tree(c, prefix, i == len(kids) - 1, is_root=False)

In [39]:
bases = ["CAPP", "CASS", "CONSTIT", "INCA", "JADE"]
schema = analyze("/mnt/data/Murphy/src", bases, split_level=0)

In [40]:
for base in bases:
    print_base(schema[base], base)

### CAPP  (1 documents)

— arbre: TEXTE_JURI_JUDI —
TEXTE_JURI_JUDI  ×1
├─ META  ×1
│  ├─ META_COMMUN  ×1
│  │  ├─ ID  ×1
│  │  ├─ ANCIEN_ID  ×1
│  │  ├─ ORIGINE  ×1
│  │  ├─ URL  ×1
│  │  └─ NATURE  ×1
│  └─ META_SPEC  ×1
│     ├─ META_JURI  ×1
│     │  ├─ TITRE  ×1
│     │  ├─ DATE_DEC  ×1
│     │  ├─ JURIDICTION  ×1
│     │  ├─ NUMERO  ×1
│     │  └─ SOLUTION  ×1
│     └─ META_JURI_JUDI  ×1
│        ├─ NUMEROS_AFFAIRES  ×1
│        │  └─ NUMERO_AFFAIRE  ×1
│        ├─ PUBLI_BULL  ×1
│        ├─ FORMATION  ×1
│        ├─ FORM_DEC_ATT  ×1
│        ├─ DATE_DEC_ATT  ×1
│        ├─ SIEGE_APPEL  ×1
│        ├─ JURI_PREM  ×1
│        ├─ LIEU_PREM  ×1
│        ├─ DEMANDEUR  ×1
│        ├─ DEFENDEUR  ×1
│        ├─ PRESIDENT  ×1
│        ├─ AVOCAT_GL  ×1
│        ├─ AVOCATS  ×1
│        └─ RAPPORTEUR  ×1
├─ TEXTE  ×1
│  ├─ BLOC_TEXTUEL  ×1
│  │  └─ CONTENU  ×1
│  │     ├─ p +  ×103  [103–103] ~103.00 ±0.00
│  │     │  └─ br *  ×6  [0–3] ~0.06 ±0.36
│  │     └─ br  ×1
│  ├─ SOMMAIRE  ×1
│  └─